# 02 — Bronze: union OneLake diagnostic JSON

For every workspace listed in `config.json → monitored_workspaces[]`, the notebook:

1. Lists all lakehouses in that workspace via the Fabric REST API.
2. Probes each lakehouse for a `Files/DiagnosticLogs/OneLake/Workspaces/` folder — the canonical layout written when OneLake diagnostics is enabled.
3. Reads hourly `PT1H.json` partitions in the lookback window from every lakehouse where that folder exists, and merges into `bronze_onelake_raw_events`.

**You only need to list workspace names** in `config.json` — the diagnostics lakehouse(s) are auto-detected.

Idempotent: dedupes on the natural key `(correlationId, accessStartTime, operationName, Resource)` within the lookback window.

In [ ]:
import json, os
from pyspark.sql import functions as F

# When run inside Fabric, the notebook resource folder contains config.json.
# Fabric exposes notebook-attached files via mssparkutils / notebookutils.
try:
    import notebookutils  # type: ignore
    cfg_path = notebookutils.nbResPath + '/builtin/config.json'
    if not os.path.exists(cfg_path):
        # Fallback: lakehouse Files/config.json
        cfg_path = '/lakehouse/default/Files/config.json'
except Exception:
    cfg_path = './config.json'

with open(cfg_path, 'r', encoding='utf-8') as f:
    CFG = json.load(f)

OBS_WS  = CFG['observability_workspace_name']
OBS_LH  = CFG['observability_lakehouse_name']
TBL     = CFG['tables']
API     = CFG['fabric_api']
# monitored_workspaces is a list of workspace display names (strings).
# Backwards-compat: also accept the old [{workspace_name: ...}] shape.
_raw_mon = CFG['monitored_workspaces']
MONITOR = [m if isinstance(m, str) else m['workspace_name'] for m in _raw_mon]
INGEST  = CFG['ingestion']
print(f'Observability workspace : {OBS_WS}')
print(f'Observability lakehouse : {OBS_LH}')
print(f'Monitored workspaces    : {MONITOR}')

## Discover diagnostics-enabled lakehouses in each monitored workspace

In [ ]:
import requests
from datetime import datetime, timedelta, timezone
import notebookutils  # type: ignore

TOKEN = notebookutils.credentials.getToken(API['token_audience'])
HEADERS = {'Authorization': f'Bearer {TOKEN}'}
BASE = API['base_url'].rstrip('/')

def fab_get(path, params=None):
    r = requests.get(f'{BASE}{path}', headers=HEADERS, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

def list_paged(path, params=None):
    items, ct = [], None
    while True:
        p = dict(params or {})
        if ct: p['continuationToken'] = ct
        data = fab_get(path, p)
        items.extend(data.get('value', []))
        ct = data.get('continuationToken')
        if not ct: break
    return items

all_workspaces = list_paged('/workspaces')
ws_by_name = {w['displayName']: w for w in all_workspaces}

DIAG_ROOT = INGEST.get('diagnostics_root_path', 'Files/DiagnosticLogs/OneLake').strip('/')

def onelake_path(ws_id, item_id, suffix=''):
    return f'abfss://{ws_id}@onelake.dfs.fabric.microsoft.com/{item_id}/{suffix}'.rstrip('/')

def has_diagnostics(ws_id, item_id):
    probe = onelake_path(ws_id, item_id, f'{DIAG_ROOT}/Workspaces')
    try:
        entries = notebookutils.fs.ls(probe)
        return len(entries) > 0
    except Exception:
        return False

diag_lakehouses = []  # list of dicts: {workspace_name, workspace_id, lakehouse_name, lakehouse_id, root}
for ws_name in MONITOR:
    ws = ws_by_name.get(ws_name)
    if not ws:
        print(f'  ! skipping workspace (not found / no access): {ws_name}')
        continue
    lakehouses = list_paged(f"/workspaces/{ws['id']}/items", {'type': 'Lakehouse'})
    enabled = []
    for lh in lakehouses:
        if has_diagnostics(ws['id'], lh['id']):
            enabled.append(lh)
            diag_lakehouses.append({
                'workspace_name': ws_name,
                'workspace_id':   ws['id'],
                'lakehouse_name': lh['displayName'],
                'lakehouse_id':   lh['id'],
                'root':           onelake_path(ws['id'], lh['id'], DIAG_ROOT),
            })
    print(f'  {ws_name}: {len(lakehouses)} lakehouse(s), {len(enabled)} with OneLake diagnostics enabled')
    for lh in enabled:
        print(f"      - {lh['displayName']}")

if not diag_lakehouses:
    print('No diagnostics-enabled lakehouses found in monitored workspaces. Nothing to ingest.')

## Build list of source paths

In [ ]:
now = datetime.now(timezone.utc)
lookback = timedelta(hours=int(INGEST['lookback_hours']))

# Build hourly partitions to scan (avoids whole-tree listing)
hours = []
t = now - lookback
while t <= now:
    hours.append(t)
    t += timedelta(hours=1)

paths = []  # list of (source_workspace_name, abfss_glob)
for d in diag_lakehouses:
    base = f"{d['root']}/Workspaces/*"  # glob over the *target* workspace id partition
    for h in hours:
        glob = (f'{base}/'
                f'y={h.year:04d}/m={h.month:02d}/d={h.day:02d}/h={h.hour:02d}/m=*/PT1H.json')
        paths.append((d['workspace_name'], glob))

print(f'Will scan {len(paths)} hourly partitions across {len(diag_lakehouses)} diagnostics lakehouse(s).')

## Read JSON, normalize, and dedupe

In [ ]:
from pyspark.sql import functions as F, types as T

def safe_read(glob):
    try:
        # OneLake / Azure diagnostic PT1H.json files are NDJSON (one event per line).
        # Do NOT set multiLine=true — that would parse each file as a single JSON
        # document and dump everything into _corrupt_record.
        return (spark.read
                    .option('recursiveFileLookup', 'false')
                    .json(glob)
                    .withColumn('_source_file', F.input_file_name()))
    except Exception as e:
        return None

frames = []
for ws_name, glob in paths:
    df = safe_read(glob)
    if df is None or df.rdd.isEmpty():
        continue
    df = df.withColumn('source_workspace_name', F.lit(ws_name))
    frames.append(df)

if not frames:
    print('No new diagnostic events in lookback window. Exiting.')
    dbutils.notebook.exit('no-data') if 'dbutils' in dir() else None
else:
    raw = frames[0]
    for f_ in frames[1:]:
        raw = raw.unionByName(f_, allowMissingColumns=True)

    # OneLake diagnostic logs use the Azure Monitor envelope: top-level
    # {category, time, resourceId, id, type, data:{...}}.  All event fields
    # live under data.* with mixed casing (executingUpn, callerIpAddress,
    # resource).  Flatten and rename to the canonical names used downstream.
    if 'data' in raw.columns:
        raw = raw.select(
            F.col('source_workspace_name'),
            F.col('_source_file'),
            F.col('time').alias('_envelope_time'),
            F.col('data.workspaceId').alias('workspaceId'),
            F.col('data.itemId').alias('itemId'),
            F.col('data.itemType').alias('itemType'),
            F.col('data.tenantId').alias('tenantId'),
            F.col('data.executingPrincipalId').alias('executingPrincipalId'),
            F.col('data.executingUpn').alias('executingUPN'),
            F.col('data.executingPrincipalType').alias('executingPrincipalType'),
            F.col('data.correlationId').alias('correlationId'),
            F.col('data.operationName').alias('operationName'),
            F.col('data.operationCategory').alias('operationCategory'),
            F.col('data.accessStartTime').alias('accessStartTime'),
            F.col('data.accessEndTime').alias('accessEndTime'),
            F.col('data.originatingApp').alias('originatingApp'),
            F.col('data.serviceEndpoint').alias('serviceEndpoint'),
            F.col('data.resource').alias('Resource'),
            F.col('data.capacityId').alias('capacityId'),
            F.col('data.httpStatusCode').alias('httpStatusCode'),
            F.col('data.isShortcut').alias('isShortcut'),
            F.col('data.accessedViaResource').alias('accessedViaResource'),
            F.col('data.callerIpAddress').alias('callerIPAddress'),
            # contentLength is not emitted by OneLake diagnostics today;
            # add as null so the bronze schema stays stable.
            F.lit(None).cast('long').alias('contentLength'),
        )

    # Defensive cast — make sure every expected column exists
    select_cols = ['workspaceId','itemId','itemType','tenantId',
        'executingPrincipalId','executingUPN','executingPrincipalType',
        'correlationId','operationName','operationCategory',
        'accessStartTime','accessEndTime','originatingApp','serviceEndpoint',
        'Resource','capacityId','httpStatusCode','isShortcut',
        'accessedViaResource','callerIPAddress','contentLength']
    for c in select_cols:
        if c not in raw.columns:
            raw = raw.withColumn(c, F.lit(None))

    norm = (raw
        .withColumn('accessStartTime', F.to_timestamp('accessStartTime'))
        .withColumn('accessEndTime',   F.to_timestamp('accessEndTime'))
        .withColumn('httpStatusCode',  F.col('httpStatusCode').cast('int'))
        .withColumn('isShortcut',      F.col('isShortcut').cast('boolean'))
        .withColumn('contentLength',   F.col('contentLength').cast('long'))
        .withColumn('event_date',      F.to_date('accessStartTime'))
        .withColumn('_ingested_at',    F.current_timestamp())
        .select('event_date','source_workspace_name', *select_cols, '_source_file','_ingested_at')
    )

    # MERGE into bronze keyed by natural key
    norm.createOrReplaceTempView('incoming_bronze')
    spark.sql(f'''
        MERGE INTO {TBL['bronze']} t
        USING (SELECT * FROM incoming_bronze) s
        ON  t.correlationId   <=> s.correlationId
        AND t.accessStartTime <=> s.accessStartTime
        AND t.operationName   <=> s.operationName
        AND t.Resource        <=> s.Resource
        WHEN NOT MATCHED THEN INSERT *
    ''')
    print('Bronze merge complete. Row count delta:')
    spark.sql(f'SELECT COUNT(*) AS rows FROM {TBL[chr(34)+chr(98)+chr(114)+chr(111)+chr(110)+chr(122)+chr(101)+chr(34)]}').show() if False else None
    print(spark.table(TBL['bronze']).count(), 'total rows in bronze')